# Week 3: training a hit filter by hand

Train the MLP hit filter with *your* training loop, look at the curves, and choose a working point.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from minihep.utils import ensure_toy_data, repo_root

ROOT = repo_root()
os.chdir(ROOT)  # every path below is relative to the repository root
sys.path.insert(0, str(ROOT))  # so `import exercises...` works from a notebook
FAST = bool(os.environ.get("MINIHEP_FAST"))  # smaller numbers for a quick check that everything runs
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(min(8, os.cpu_count() or 1))
print("device:", DEVICE, "| torch", torch.__version__, "| fast mode" if FAST else "")

In [ ]:
from torch import nn

from exercises.week03.ex02_training_loop import LRConfig, evaluate, train
from minihep.data.dataset import ToyTrackingDataset
from minihep.models.dense import Dense
from minihep.models.hitfilter import HitFilter
from minihep.models.input import InputNet
from minihep.models.tasks import HitFilterTask

data_dir = ensure_toy_data(num_train=200 if FAST else 2000, num_val=50 if FAST else 200)
fields = ["x", "y", "z", "r", "eta", "phi"]
cfg = {"inputs": {"hit": fields}, "targets": {"hit": ["on_valid_particle"]}}
train_ds = ToyTrackingDataset(str(data_dir / "train"), **cfg)
val_ds = ToyTrackingDataset(str(data_dir / "val"), **cfg)


def make_mlp(dim=64):
    net = InputNet("hit", Dense(len(fields), dim, hidden_layers=[dim, dim]), fields)
    return HitFilter(nn.ModuleList([net]), nn.ModuleList([HitFilterTask("hit_filter", "hit", "on_valid_particle", dim)]))


model = make_mlp()
print(model)
print(sum(p.numel() for p in model.parameters()), "parameters")

In [ ]:
steps = 300 if FAST else 3000
history = train(model, train_ds, num_steps=steps, cfg=LRConfig(initial=1e-4, max=2e-3, end=1e-5, pct_start=0.05), device=DEVICE)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
smooth = np.convolve(history.loss, np.ones(50) / 50, mode="valid")
axes[0].plot(history.loss, alpha=0.3)
axes[0].plot(smooth)
axes[0].set_title("loss")
axes[1].plot(history.lr)
axes[1].set_title("learning rate (OneCycleLR)")
axes[2].plot(history.grad_norm, alpha=0.5)
axes[2].set_yscale("log")
axes[2].set_title("gradient norm (before clipping)")
plt.show()
print(evaluate(model, val_ds, "hit_filter", device=DEVICE))

## Choosing a threshold

The filter's job is to throw away noise and uninteresting hits *without losing hits the tracker needs*.
So we want very high recall, then as much precision as possible. hepattn's TrackML filter uses a
threshold of 0.1 (600 MeV) or 0.314 (900 MeV), chosen exactly this way.

In [ ]:
@torch.no_grad()
def collect(model, ds):
    model.eval()
    probs, truth = [], []
    for i in range(len(ds)):
        inputs, targets = ds[i]
        out = model({k: v.to(DEVICE) for k, v in inputs.items()})
        probs.append(out["final"]["hit_filter"]["hit_logit"].sigmoid().cpu()[0])
        truth.append(targets["hit_on_valid_particle"][0])
    model.train()
    return torch.cat(probs), torch.cat(truth)


probs, truth = collect(model, val_ds)
thresholds = np.linspace(0.01, 0.99, 99)
recall = [((probs >= t) & truth).sum().item() / truth.sum().item() for t in thresholds]
kept = [(probs >= t).float().mean().item() for t in thresholds]
plt.plot(thresholds, recall, label="recall (fraction of good hits kept)")
plt.plot(thresholds, kept, label="fraction of all hits kept")
plt.xlabel("threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Your turn

1. Why can an MLP that sees one hit at a time not tell a 0.9 GeV particle from a 1.1 GeV one? Which
   information would it need? (Next week's transformer gets it.)
2. Retrain with `loss_fn="focal"` in the task and compare the threshold plot.
3. Set `grad_clip=None` and a 10x higher max learning rate. What happens to the gradient norm plot?